[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/23-serving.ipynb)

# Module 23: Serving

Companion notebook to [Module 23](https://transformers-and-math.drmhse.com/modules/23-serving) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import time
import torch
torch.manual_seed(0)

## Part 1 — The memory wall (section 23.1)

In [ ]:
def kv_bytes(layers, kv_heads, d_head, ctx, batch=1, nbytes=2):
    return 2 * layers * kv_heads * d_head * ctx * batch * nbytes

print(f'LLaMA-2 7B, 4,096 tokens, 1 sequence:  {kv_bytes(32, 32, 128, 4096) / 2 ** 30:.1f} GiB')
print(f'LLaMA-2 7B, 4,096 tokens, 32 sequences: {kv_bytes(32, 32, 128, 4096, 32) / 2 ** 30:.1f} GiB')
print(f'LLaMA-3 8B, 8,192 tokens, 1 sequence:  {kv_bytes(32, 8, 128, 8192) / 2 ** 30:.1f} GiB')

## Part 2 — Prefill is compute-bound, decode is memory-bound (section 23.2)
The same weight matrix, applied to 1 token at a time and to many at once. Time per token falls sharply with the batch, because reading the weights is the cost that gets shared.

In [ ]:
d = 2048
W = torch.randn(d, d)
for n_tokens in (1, 8, 64, 512):
    x = torch.randn(n_tokens, d)
    for _ in range(3): x @ W                            # warm up
    start = time.perf_counter()
    for _ in range(20): x @ W
    per_token = (time.perf_counter() - start) / 20 / n_tokens
    print(f'{n_tokens:4} tokens at once: {per_token * 1e6:8.2f} µs per token   (each weight read is used {n_tokens}×)')

## Part 3 — Speculative sampling is exact (section 23.5)
Draft from $q$, accept with $\min(1, p/q)$, otherwise resample from the leftover. The emitted tokens follow $p$ exactly.

In [ ]:
p = torch.tensor([0.05, 0.40, 0.30, 0.15, 0.10])
q = torch.tensor([0.30, 0.10, 0.30, 0.10, 0.20])
n = 200_000
draft = torch.multinomial(q, n, replacement=True)
accept = torch.rand(n) < torch.clamp(p[draft] / q[draft], max=1)
leftover = torch.clamp(p - q, min=0); leftover /= leftover.sum()
out = torch.where(accept, draft, torch.multinomial(leftover, n, replacement=True))
print('emitted frequencies:', [round(v, 3) for v in (torch.bincount(out, minlength=5) / n).tolist()])
print('target p:           ', [round(v, 3) for v in p.tolist()])
print(f'acceptance rate {accept.float().mean():.3f}  (predicted Σ min(p, q) = {torch.minimum(p, q).sum():.3f})')

alpha, gamma = 0.8, 4
print(f'expected tokens per large-model pass at α={alpha}, γ={gamma}: {(1 - alpha ** (gamma + 1)) / (1 - alpha):.2f}')

## Your turn
1. How many 4,096-token LLaMA-3-8B sequences fit in 64 GiB of cache memory?
2. In Part 3, make `q` equal to `p`. What is the acceptance rate, and why?